### 🌸 FLY Academy — Visual Themes and Documentation

The project documentation offers **Rose 🌸, Onyx Black 🖤, and King Green 🌿** appearances at [FLY Academy](https://arharif.github.io/Fly-collaborative-travel-agent/). The interactive Academy and its footer are separate from this Python backend. [About X1 · @arharif](https://arharif.github.io/about-x1).

**Security:** Never paste AWS or Hugging Face credentials into shared notebook cells.


# ✈️ FLY · From Zero to Hugging Face PRO

### An executable, beginner-friendly Google Colab deployment lab

**What this notebook does:** obtains the exact repository code, installs dependencies, verifies AWS/DeepSeek V3.2, launches the pink Fly UI locally, and publishes the standalone Gradio app to your own Hugging Face PRO Space.

**Before you begin:** Read [`docs/TUTORIAL_FROM_ZERO.md`](../docs/TUTORIAL_FROM_ZERO.md) for AWS IAM and secrets setup. **Never paste or print passwords, tokens or AWS keys into cells**. Runtime: Colab **CPU** is enough.

**Important:** This notebook uses the same `space/app.py` as deployment to minimize local/hosted differences. It does **not** replace your professor's baseline notebook; reference Cell 14/16 files are included in this repository.

## 00 — Get your project code into this Colab session

Choose **GitHub mode** if you've uploaded this package to a *public* GitHub repository. Fill in your actual GitHub username and repository name below. Otherwise choose **ZIP mode** and upload the **complete GitHub project ZIP** from your computer (not the smaller Hugging Face Space-only ZIP).

You only have to change the first three values. Each new Colab runtime needs this step.

In [ ]:
# CELL 00 — GET THE PROJECT SOURCE
from pathlib import Path
import subprocess, zipfile

SOURCE_MODE = "zip"  # "zip" (simplest) or "github" (public repo)
GITHUB_USERNAME = "REPLACE_WITH_YOUR_GITHUB_USERNAME"
GITHUB_REPOSITORY = "fly-collaborative-travel-agent"

PROJECT_ROOT = None
if SOURCE_MODE == "github":
    if "REPLACE_" in GITHUB_USERNAME:
        raise ValueError("Set GITHUB_USERNAME above, or use SOURCE_MODE='zip'.")
    target = Path("/content/fly-repo")
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}.git"
    if not (target / "space" / "app.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)
    PROJECT_ROOT = target
elif SOURCE_MODE == "zip":
    from google.colab import files
    print("Choose your complete FLY GitHub project ZIP file.")
    uploaded = files.upload()
    zips = [n for n in uploaded if n.lower().endswith(".zip")]
    if len(zips) != 1:
        raise RuntimeError("Select exactly one GitHub project ZIP.")
    destination = Path("/content/fly-uploaded")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as archive:
        for member in archive.infolist():
            # Prevent dangerous archive paths when extracting unknown ZIPs.
            resolved = (destination / member.filename).resolve()
            if not resolved.is_relative_to(destination.resolve()):
                raise RuntimeError("ZIP contained an unsafe extraction path")
        archive.extractall(destination)
    roots = sorted({p.parent.parent for p in destination.rglob("space/app.py")})
    if len(roots) != 1:
        raise RuntimeError("Expected exactly one project folder containing space/app.py")
    PROJECT_ROOT = roots[0]
else:
    raise ValueError("SOURCE_MODE must be 'zip' or 'github'")

SPACE_PATH = PROJECT_ROOT / "space"
for expected in ["app.py", "agent_backend.py", "requirements.txt", "README.md"]:
    assert (SPACE_PATH / expected).is_file(), f"Missing {expected}"
print("SUCCESS: Project files are ready")
print("Project path:", PROJECT_ROOT)
print("Files:", [p.name for p in SPACE_PATH.iterdir() if p.is_file()])

## 01 — Install the exact library versions used by the Space

This is not a GPU-training project. It uses a remote AWS model, so CPU Colab is sufficient. This cell installs packages into your **current Colab runtime**. Restarting the runtime means you'll need to run it again.

In [ ]:
# CELL 01 — INSTALL DEPENDENCIES
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(SPACE_PATH / "requirements.txt"),
    "huggingface_hub>=0.32,<2",
])
import gradio as gr, smolagents, litellm, boto3
print("SUCCESS: Dependencies installed")
print("Gradio:", gr.__version__)
print("Smolagents:", getattr(smolagents, "__version__", "installed"))

## 02 — Load AWS secrets securely

In Colab, click the **key/Secrets icon** in the left sidebar. Add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`; enable notebook access. If credentials are temporary, add `AWS_SESSION_TOKEN` too. The code below prints **only status**, never values.

In [ ]:
# CELL 02 — AWS CONFIGURATION
import os, boto3
from google.colab import userdata

access_key = userdata.get("AWS_ACCESS_KEY_ID")
secret_key = userdata.get("AWS_SECRET_ACCESS_KEY")
if not access_key or not secret_key:
    raise RuntimeError("Add AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY to Colab Secrets.")
os.environ["AWS_ACCESS_KEY_ID"] = access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = secret_key
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
os.environ["AWS_REGION"] = "us-east-1"
# If you use temporary credentials, add AWS_SESSION_TOKEN as another Colab secret.
try:
    session_token = userdata.get("AWS_SESSION_TOKEN")
except Exception:
    session_token = None
if session_token:
    os.environ["AWS_SESSION_TOKEN"] = session_token
else:
    os.environ.pop("AWS_SESSION_TOKEN", None)
print("SUCCESS: AWS configuration loaded without displaying credentials.")

## 03 — Test AWS credentials independently of the model

The STS identity request tests that AWS accepts your credential pair. It does *not* yet prove permission to invoke the model.

In [ ]:
# CELL 03 — AWS AUTHENTICATION / STS
session = boto3.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    aws_session_token=os.getenv("AWS_SESSION_TOKEN"),
    region_name="us-east-1",
)
identity = session.client("sts").get_caller_identity()
print("SUCCESS: AWS identity authenticated.")
print("Identity verified (account number intentionally not displayed).")

## 04 — First small DeepSeek V3.2 call (billable)

This checks IAM authorization and actual AWS model availability. In this project `deepseek.v3.2` is the AWS ID; `bedrock/deepseek.v3.2` is the LiteLLM provider-qualified ID. If the request is denied, review AWS IAM and model onboarding in the tutorial.

In [ ]:
# CELL 04 — DIRECT DEEPSEEK V3.2 TEST
import litellm
litellm.drop_params = True
response = litellm.completion(
    model="bedrock/deepseek.v3.2",
    messages=[{"role":"user","content":"In one sentence, what is an AI agent?"}],
    max_tokens=140,
)
answer = response.choices[0].message.content or "(No text returned)"
print("SUCCESS: Model returned:")
print(answer[:700])

## 05 — Create your Smolagents backend from the exact deployed source

In `agent_backend.py`, `stop=REMOVE_PARAMETER` avoids the unsupported Bedrock `stopSequences` request field. The agent can optionally search/read websites. A fresh agent is created per request; Gradio holds conversation context in `gr.State`.

In [ ]:
# CELL 05 — VERIFY THE AGENT BACKEND
import sys
if str(SPACE_PATH) not in sys.path:
    sys.path.insert(0, str(SPACE_PATH))
from agent_backend import make_collaborative_travel_agent, assignment_model
agent = make_collaborative_travel_agent()
print("SUCCESS: Agent initialized")
print("Model:", assignment_model.model_id)
print("Tools:", sorted(agent.tools.keys()))
print("Max steps:", agent.max_steps)
print("Model-specific stop omission:", "stop" in assignment_model.kwargs)

## 06 — Launch the pink Fly chatbot in your existing Colab

This imports the **same** UI source that runs on Hugging Face. Use **Preview (no AWS)** first, then Generate and Send. The local Gradio link is temporary. **Do not leave the public link online unattended** because AI requests bill your AWS account.

In [ ]:
# CELL 06 — LAUNCH GRADIO FROM THE HOSTED APP SOURCE
import runpy
app_context = runpy.run_path(str(SPACE_PATH / "app.py"), run_name="fly_colab_test")
fly_demo = app_context["fly_demo"]
FLY_CSS = app_context["FLY_CSS"]
fly_demo.launch(
    share=True,
    debug=False,
    prevent_thread_lock=True,
    show_error=False,
    theme=gr.themes.Soft(),
    css=FLY_CSS,
)
print("Fly local demo launched. Colab can continue to the next cells.")

## 07 — Authenticate with Hugging Face PRO

Save your HF fine-grained write token in **Colab Secrets → HF_TOKEN**, notebook access on. Verify the returned username is the account with the PRO entitlement. Do **not** paste the token into code.

In [ ]:
# CELL 07 — HUGGING FACE AUTHENTICATION
from huggingface_hub import HfApi
hf_token = userdata.get("HF_TOKEN")
if not hf_token or not hf_token.startswith("hf_"):
    raise RuntimeError("Create a fine-grained HF token with write permission and save it as HF_TOKEN.")
hf_api = HfApi(token=hf_token)
hf_username = hf_api.whoami()["name"]
hf_repo_id = f"{hf_username}/haii-fly-travel-live"
print("SUCCESS: Hugging Face authenticated")
print("Account:", hf_username)
print("Target Space:", hf_repo_id)

## 08 — Publish the real Gradio Space (not Static HTML)

The code uploads **only** the four files inside `space/`, so `app.py` ends up at the root of the Hugging Face Space. If the Space doesn't exist, it creates a private Gradio Space. It refuses to upload to an existing Static Space. It never uploads credentials. **Before Generate can work** you must configure AWS Secrets and Region under Space Settings separately.

In [ ]:
# CELL 08 — CREATE / UPLOAD LIVE GRADIO SPACE
import re
readme = (SPACE_PATH / "README.md").read_text(encoding="utf-8")
match = re.search(r"^short_description:\s*(.+)$", readme, re.MULTILINE)
assert match and len(match.group(1).strip()) <= 60, "README metadata invalid"
assert re.search(r"^sdk:\s*gradio\s*$", readme, re.MULTILINE), "Must be sdk: gradio"

# Check pre-existing Space first, before writing anything.
try:
    space_info = hf_api.space_info(hf_repo_id)
    if space_info.sdk != "gradio":
        raise RuntimeError(f"Space {hf_repo_id} is {space_info.sdk}, not gradio. Choose another repo ID.")
except Exception as exc:
    from huggingface_hub.utils import RepositoryNotFoundError
    if not isinstance(exc, RepositoryNotFoundError):
        raise
    hf_api.create_repo(
        repo_id=hf_repo_id,
        repo_type="space",
        space_sdk="gradio",
        private=True,
        exist_ok=False,
    )

hf_api.upload_folder(
    repo_id=hf_repo_id,
    repo_type="space",
    folder_path=str(SPACE_PATH),
    commit_message="Deploy Fly conversational Gradio agent",
    allow_patterns=["app.py", "agent_backend.py", "requirements.txt", "README.md"],
)
print("SUCCESS: Uploaded Gradio application source.")
print(f"Space URL: https://huggingface.co/spaces/{hf_repo_id}")
print("Next: add Space AWS secrets in Settings. Do not upload credentials.")

## 09 — Configure Space Secrets manually, then inspect its status

**On Hugging Face:** Space → Settings → Variables and secrets. Create secrets `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `FLY_APP_PASSWORD`; optional `FLY_APP_USER` and (for temporary credentials) `AWS_SESSION_TOKEN`. Add a **Variable** `AWS_REGION=us-east-1`. The Space rebuilds/restarts after settings changes. Keep it private while testing. Then run this status cell.

In [ ]:
# CELL 09 — REMOTE SPACE VERIFICATION
space_info = hf_api.space_info(hf_repo_id)
repo_files = set(hf_api.list_repo_files(hf_repo_id, repo_type="space"))
expected = {"app.py", "agent_backend.py", "requirements.txt", "README.md"}
assert expected.issubset(repo_files), f"Missing files: {expected - repo_files}"
assert space_info.sdk == "gradio", "Wrong Space SDK"
print("SUCCESS: Source files verified on HF")
print("URL:", f"https://huggingface.co/spaces/{hf_repo_id}")
try:
    runtime = hf_api.get_space_runtime(hf_repo_id)
    print("Runtime stage:", getattr(runtime, "stage", "unknown"))
    print("Hardware:", getattr(runtime, "hardware", "unknown"))
except Exception as exc:
    print("Runtime lookup not available:", type(exc).__name__)
print("Next in Space: test Preview, then Design my journey, then Send.")

## 10 — Capture scientific evaluation evidence

In your original professor notebook, keep the baseline agent (Cells 9/10) unchanged. For each scenario run the same user query with the baseline and the improved agent. Preserve **real** transcripts, record question count, rationale quality, and correction handling. Do not count Preview's fabricated sample plan as empirical AI evidence.

**Example tests:** (1) four quiet days in Kyoto; (2) countryside choice based on price, then request central location; (3) change Day 2 to quieter and cheaper; (4) explicit revision that supersedes an inferred preference.

The full worksheet and rubric mapping are in `docs/HAI_EVALUATION.md`.

In [ ]:
# CELL 10 — PRINT YOUR EVALUATION CHECKLIST
checks = [
    "Save an authentic baseline and improved output for the SAME Kyoto prompt",
    "Count unsolicited questions and preference-based explanations",
    "Run the same explicit correction in each version",
    "Test Custom duration and Other interests",
    "Verify a relevant place link and observe photo fallback",
    "Record errors and limitations rather than hiding them",
]
for i, item in enumerate(checks, 1):
    print(f"{i}. [ ] {item}")
print("Next: Share the GitHub repo and the Space URL with your professor as appropriate.")

## Finish — What you have built

You have now followed the chain **IAM → Bedrock → LiteLLM → Smolagents → Gradio → Hugging Face PRO**, and have both a GitHub source package and a publishable interactive docs website. For backend flow diagrams, detailed debugging and official references, see the docs folder in your repository. AWS inference and Hugging Face services can change or fail externally; monitor costs and verify real behavior. Never share credentials in your submission.